<a href="https://colab.research.google.com/github/victorgarciiag/DigitalScope/blob/main/digitalscope_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instalación de librerías necesarias
!pip install supabase python-dotenv requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 4.0 MB/s eta 0:00:00


In [3]:
from google.colab import userdata
from supabase import create_client

# Cargamos las credenciales desde los Secrets de Colab
SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_KEY = userdata.get('SUPABASE_KEY')
GOOGLE_PLACES_KEY = userdata.get('GOOGLE_PLACES_KEY')

# Creamos el cliente de Supabase
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

print("✅ Conexión a Supabase establecida correctamente")

✅ Conexión a Supabase establecida correctamente


In [4]:
import requests

def buscar_negocios(query, ciudad, api_key, max_resultados=20):
    """
    Busca negocios en Google Places API (New)
    query: tipo de negocio ("peluquería", "clínica estética"...)
    ciudad: ciudad donde buscar
    """
    url = "https://places.googleapis.com/v1/places:searchText"

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": api_key,
        "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.nationalPhoneNumber,places.websiteUri,places.rating,places.userRatingCount,places.primaryTypeDisplayName"
    }

    body = {
        "textQuery": f"{query} en {ciudad}",
        "maxResultCount": max_resultados,
        "languageCode": "es"
    }

    response = requests.post(url, headers=headers, json=body)

    if response.status_code == 200:
        data = response.json()
        print(f"✅ Encontrados {len(data.get('places', []))} negocios")
        return data.get("places", [])
    else:
        print(f"❌ Error: {response.status_code} - {response.text}")
        return []

# Primera búsqueda real
negocios = buscar_negocios("peluquería", "Sevilla", GOOGLE_PLACES_KEY)

✅ Encontrados 20 negocios


In [5]:
import json

# Vemos el primer negocio para entender la estructura
print(json.dumps(negocios[0], indent=2, ensure_ascii=False))

{
  "id": "ChIJO3SVNaJuEg0R34nQ9GfDhFE",
  "nationalPhoneNumber": "955 13 93 47",
  "formattedAddress": "Av. Eduardo Dato, 43, 41018 Sevilla, España",
  "rating": 4.8,
  "websiteUri": "https://www.facebook.com/glamsevilla",
  "userRatingCount": 1043,
  "displayName": {
    "text": "Peluquería GLAM Sevilla",
    "languageCode": "es"
  },
  "primaryTypeDisplayName": {
    "text": "Peluquería",
    "languageCode": "es"
  }
}


In [7]:
def limpiar_negocio(negocio_raw, ciudad, categoria):
    """
    Transforma el formato de Google Places al formato de nuestra tabla
    """
    return {
        "place_id": negocio_raw.get("id"),
        "name": negocio_raw.get("displayName", {}).get("text"),
        "address": negocio_raw.get("formattedAddress"),
        "phone": negocio_raw.get("nationalPhoneNumber"),
        "website": negocio_raw.get("websiteUri"),
        "rating": negocio_raw.get("rating"),
        "reviews_count": negocio_raw.get("userRatingCount"),
        "category": categoria,
        "city": ciudad
    }

def guardar_negocios(negocios_raw, ciudad, categoria):
    """
    Guarda la lista de negocios en Supabase
    """
    guardados = 0
    duplicados = 0

    for negocio_raw in negocios_raw:
        negocio = limpiar_negocio(negocio_raw, ciudad, categoria)

        try:
            supabase.table("businesses").upsert(
                negocio,
                on_conflict="place_id"
            ).execute()
            guardados += 1
        except Exception as e:
            print(f"❌ Error guardando {negocio['name']}: {e}")

    print(f"✅ Guardados: {guardados} negocios")
    print(f"🔄 Duplicados ignorados: {duplicados}")

# Guardamos los 20 negocios en Supabase
guardar_negocios(negocios, "Sevilla", "Peluquería")

✅ Guardados: 20 negocios
🔄 Duplicados ignorados: 0
